In [ ]:
from mockfactory import RandomCutskyCatalog, setup_logging, Catalog
import desimodel.footprint
from mpytools import Catalog
import matplotlib.pyplot as plt
# os.environ['DESI_LOGLEVEL'] = 'ERROR'
import fitsio
from mpi4py import MPI
import logging
import numpy as np
import pandas as pd


In [ ]:
from fiber_assignment import run_FA
from mockfactory import Catalog
cutsky_lrg = Catalog.read('/pscratch/sd/e/epaillas/acm/dr2/hods/cutsky/v0.0/c000_ph000/LRG_NGC_hod000.dat.fits')

# Select a small region for testing
mask = (cutsky_lrg['RA']<205) &(cutsky_lrg['RA']> 175) & (cutsky_lrg['DEC']>35) & (cutsky_lrg['DEC']<45)
cutsky_lrg = cutsky_lrg[mask]


In [ ]:
cutsky_FA = run_FA(cutsky_lrg, release='Y3', program='dark', npasses=1, add_random_tracers=True, tracer='LRG', plot_output=False, path_to_save=None)

In [ ]:
import skyproj
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

sp = skyproj.DESSkyproj(ax=ax, extent=[178,209, 33, 48], fontsize=8)

sp.scatter(cutsky_lrg['RA'], cutsky_lrg['DEC'], s=0.001, c='k')
sp.scatter(cutsky_FA['RA'][cutsky_FA['OBS_PASS'].T[0]], cutsky_FA['DEC'][cutsky_FA['OBS_PASS'].T[0]], s=0.01,c='r') 

sp.scatter(0,0, s=10,c='r', label='Pass 1') 




sp.legend()
fig.tight_layout()

In [ ]:
from mockfactory import RandomCutskyCatalog
from mockfactory.desi import is_in_desi_footprint
from mockfactory import TabulatedRadialMask, DistanceToRedshift
from cosmoprimo.fiducial import DESI



#Create random
cutsky_rd = RandomCutskyCatalog(rarange=(cutsky_lrg['RA'].min(), cutsky_lrg['RA'].max()), decrange=(cutsky_lrg['DEC'].min(), cutsky_lrg['DEC'].max()), drange=(cutsky_lrg['Distance'].min(), cutsky_lrg['Distance'].max()), nbar=1000, seed=55)

cosmo = DESI()
distance_to_redshift = DistanceToRedshift(distance=cosmo.comoving_radial_distance)
idx = cutsky_lrg['Z'].argsort()
nz = np.interp(np.linspace(cutsky_FA['Z'].min(), cutsky_FA['Z'].max(), 100), cutsky_FA['Z'][idx], cutsky_FA['NZ'][idx])
mask_radial = TabulatedRadialMask(z=np.linspace(cutsky_lrg['Z'].min(), cutsky_lrg['Z'].max(), len(nz)), nbar=nz, interp_order=2,
                                                    norm=None)
cutsky_rd = cutsky_rd[mask_radial(distance_to_redshift(cutsky_rd['Distance']), seed=55)]

cutsky_rd = cutsky_rd[is_in_desi_footprint(cutsky_rd['RA'], cutsky_rd['DEC'], release='Y3', program='dark')]

#FA is run on random to get the objects that can be reach by a fiber (AVAILABLE column)
cutsky_FA_rd = run_FA(cutsky_rd, release='Y3', program='dark', npasses=1, add_random_tracers=False, tracer='LRG', plot_output=False, path_to_save=None)

In [ ]:
# Plot distance distribution 
plt.hist(cutsky_FA_rd['Distance'], bins=50, density=True, histtype='step', label='Randoms')
plt.hist(cutsky_FA['Distance'], bins=50, density=True, histtype='step', label='LRG')
plt.legend()
plt.show()

In [ ]:
# Plot distance distribution for AVAILABLE objects only 
plt.hist(cutsky_FA_rd['Distance'][cutsky_FA_rd['AVAILABLE']], bins=50, density=True, histtype='step', label='Randoms')
plt.hist(cutsky_FA['Distance'][cutsky_FA['AVAILABLE']], bins=50, density=True, histtype='step', label='LRG')
plt.legend()
plt.show()

In [ ]:
import skyproj
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
# sp.draw_des(label='DES', edgecolor='k')
sp = skyproj.DESSkyproj(ax=ax, extent=[178,209, 33, 48], fontsize=8)

sp.scatter(cutsky_rd['RA'], cutsky_rd['DEC'], s=0.001, c='k')
sp.scatter(cutsky_FA_rd['RA'][cutsky_FA_rd['AVAILABLE']], cutsky_FA_rd['DEC'][cutsky_FA_rd['AVAILABLE']], s=0.01,c='r') 
sp.scatter(cutsky_FA['RA'][cutsky_FA['AVAILABLE']], cutsky_FA['DEC'][cutsky_FA['AVAILABLE']], s=0.01,c='b') 
# sp.scatter(cutsky_FA_rd['RA'][cutsky_FA_rd['OBS_PASS'].T[0]], cutsky_FA_rd['DEC'][cutsky_FA_rd['OBS_PASS'].T[0]], s=0.01,c='r') 
# sp.scatter(cutsky_FA['RA'][cutsky_FA['OBS_PASS'].T[0]], cutsky_FA['DEC'][cutsky_FA['OBS_PASS'].T[0]], s=0.01,c='b') 

sp.scatter(0,0, s=10,c='r', label='cutsky mock') 
sp.scatter(0,0, s=10,c='b', label='random') 



sp.legend()
fig.tight_layout()

In [ ]:
import os 
from pycorr import TwoPointCorrelationFunction, setup_logging
import numpy as np
from mockfactory import Catalog
from matplotlib import pyplot as plt
import argparse
import logging
from mpi4py import MPI


logger = logging.getLogger('F.A.')
setup_logging()

def get_2PCF_from_LC(cat, rd_cat, edges, nthreads=256, R1R2=None, weights=None,weights_rd=None, mask=None, mask_rd=None):

    if mask is None:
        mask = np.ones(cat.csize, dtype=bool)
    else:
        if weights is not None:
            weights = weights[mask]
    if mask_rd is None:
        mask_rd = np.ones(rd_cat.csize, dtype=bool)
    else:
        if weights_rd is not None:
            weights_rd = weights_rd[mask_rd]

    result = TwoPointCorrelationFunction('smu', edges,
                                         data_positions1=[cat['RA'][mask], cat['DEC'][mask], cat['Distance'][mask]],
                                         data_positions2=None, engine='corrfunc', position_type='rdd', data_weights1=weights,
                                         randoms_positions1=[rd_cat['RA'][mask_rd], rd_cat['DEC'][mask_rd], rd_cat['Distance'][mask_rd]], randoms_weights1=weights_rd,
                                         boxsize=None, nthreads=nthreads, R1R2=R1R2)

    return result

def get_wp_from_LC(cat, rd_cat, edges, nthreads=256, R1R2=None, weights=None,weights_rd=None, mask=None, mask_rd=None):

    if mask is None:
        mask = np.ones(cat.csize, dtype=bool)
    else:
        if weights is not None:
            weights = weights[mask]
    if mask_rd is None:
        mask_rd = np.ones(rd_cat.csize, dtype=bool)
    else:
        if weights_rd is not None:
            weights_rd = weights_rd[mask_rd]

    result = TwoPointCorrelationFunction('rppi', edges,
                                         data_positions1=[cat['RA'][mask], cat['DEC'][mask], cat['Distance'][mask]],
                                         data_positions2=None, engine='corrfunc', position_type='rdd', data_weights1=weights,
                                         randoms_positions1=[rd_cat['RA'][mask_rd], rd_cat['DEC'][mask_rd], rd_cat['Distance'][mask_rd]], randoms_weights1=weights_rd,
                                         boxsize=None, nthreads=nthreads, R1R2=R1R2)


    return result

In [ ]:
from run_pc_for_FA_mocks import get_2PCF_from_LC, get_wp_from_LC

edges = [np.geomspace(0.01, 50, 48), np.linspace(-40, 40, 81)]


result_rppi_avail = get_wp_from_LC(cutsky_FA, cutsky_FA_rd, edges=edges, nthreads=256, mask=cutsky_FA['AVAILABLE'], mask_rd=cutsky_FA_rd['AVAILABLE'])

rp_lcav, wp_lcav = result_rppi_avail(return_sep=True, pimax=40)

result_rppi_FA = get_wp_from_LC(cutsky_FA, cutsky_FA_rd, edges=edges, nthreads=256, mask=(cutsky_FA['NUMOBS']>0), mask_rd=cutsky_FA_rd['AVAILABLE'], R1R2=result_rppi_avail.R1R2)

rp_fa, wp_fa = result_rppi_FA(return_sep=True, pimax=40)

result_rppi_FA_wcomp = get_wp_from_LC(cutsky_FA, cutsky_FA_rd, edges=edges, nthreads=256, mask=(cutsky_FA['NUMOBS']>0), mask_rd=cutsky_FA_rd['AVAILABLE'], R1R2=result_rppi_avail.R1R2, weights=cutsky_FA['COMP_WEIGHT'])
rp_fa_wcomp, wp_fa_wcomp = result_rppi_FA_wcomp(return_sep=True, pimax=40)

In [ ]:
plt.semilogx(rp_lcav, wp_lcav*rp_lcav, label='Available for FA')
# plt.semilogx(rp_lc, wp_lc*rp_lc, label='all')

plt.semilogx(rp_fa, wp_fa*rp_fa, label='Observed after FA')
plt.semilogx(rp_fa_wcomp, wp_fa_wcomp*rp_fa_wcomp, label='Observed after FA w/ comp weight')

plt.legend()
plt.xlabel('rp [Mpc/h]')
plt.ylabel('$r_p \cdot w_p$ [Mpc/$h$]')


# Create random desi NGC catalog

In [ ]:
from mockfactory import RandomCutskyCatalog
from mockfactory.desi import is_in_desi_footprint

custsky_rd = RandomCutskyCatalog(rarange=(85, 300), decrange=(-15, 90), drange=(0, 3000), nbar=1000, seed=42)
is_in_desi = is_in_desi_footprint(custsky_rd['RA'], custsky_rd['DEC'], release='Y3', program='dark')
custsky_rd_in_desi = custsky_rd[is_in_desi]


In [ ]:
from mockfactory import TabulatedRadialMask, DistanceToRedshift
from cosmoprimo.fiducial import DESI
import numpy as np 
def _apply_radial_mask(catalog, nz, zmin=0., zmax=6., seed=42, norm=None):
    print('Applying radial mask.')
    from mockfactory import TabulatedRadialMask, DistanceToRedshift
    from cosmoprimo.fiducial import DESI
    cosmo = DESI()
    distance_to_redshift = DistanceToRedshift(distance=cosmo.comoving_radial_distance)
    nz_filename = '/global/cfs/cdirs/desi/survey/catalogs/Y1/LSS/iron/LSScats/v1.5/LRG_NGC_nz.txt'
    zbin_min, zbin_max, n_z = np.genfromtxt(nz_filename, usecols=(1, 2, 3)).T
    mask_radial = TabulatedRadialMask(z=np.linspace(zbin_min[0], zbin_max[-1], len(n_z)), nbar=n_z, interp_order=2,
                                                    norm=norm)
    return catalog[mask_radial(distance_to_redshift(catalog['Distance']), seed=seed)]

custsky_rd_in_desi = _apply_radial_mask(custsky_rd_in_desi, nz=None, zmin=0., zmax=6., seed=42, norm=None)

In [ ]:
#Plot distance distribution
from matplotlib import pyplot as plt
plt.hist(custsky_rd_in_desi['Distance'], bins=50)
plt.show()

In [ ]:
# To show on a healpix map 
# to install CRStools:
# git clone https://github.com/4most-crs/4MOST_CRS_tools.git 
# cd 4MOST_CRS_tools
# python -m pip install -e . --user

from CRStools import utils 
hpmap = utils.create_hp_map(custsky_rd_in_desi['RA'], custsky_rd_in_desi['DEC'], nside=256)
utils.plot_moll(hpmap, figsize=(10,8), desi_footprint=True)


## Compute 2PCF comparison 

### You can run FA for mock and random on a single node using the script run_FA_for_mock.py it should take few minutes.

```
srun -n 64 python run_FA_for_mock.py --input_catalog /pscratch/sd/e/epaillas/acm/dr2/hods/cutsky/v0.0/c000_ph000/LRG_NGC_hod000.dat.fits --output_catalog /pscratch/sd/a/arocher/LRG_NGC_hod000_with_FA.dat.fits --npasses 7 --release Y3 --program dark 
srun -n 64 python run_FA_for_mock.py --input_catalog /pscratch/sd/a/arocher/randoms_LRG_NGC_hod000_nz.dat.fits --output_catalog /pscratch/sd/a/arocher/randoms_LRG_NGC_hod000_with_FA_nz.dat.fits --npasses 7 --release Y3 --program dark 
```

### Then you can compute the 2PCF using the script run_pc_for_FA_mocks.py or just run it in the nb (~few minutes)
```srun -N 1 python run_pc_for_FA_mocks.py --input_catalog /pscratch/sd/a/arocher/LRG_NGC_hod000_with_FA.dat.fits --input_rd_catalog /pscratch/sd/a/arocher/randoms_LRG_NGC_hod000_with_FA_nz.dat.fits ```

Output correlations and plots will be saved in a new repository 2PCF_FA_results. You can give the output directory path using the command --output_dir  

In [ ]:
from run_pc_for_FA_mocks import get_2PCF_from_LC, get_wp_from_LC
# After having run FA on the cutsky and random mocks

cutsky = Catalog.read('/pscratch/sd/a/arocher/LRG_NGC_hod000_with_FA.dat.fits')
if 'AVAILABLE' not in cutsky.columns():
    raise ValueError('The cutsky catalog must have an AVAILABLE column indicating which targets were assigned fibers')

mask_tr = (cutsky['TRACER'] == 'LRG')
cutsky_rd = Catalog.read('/pscratch/sd/a/arocher/randoms_LRG_NGC_hod000_with_FA_nz.dat.fits')

edges_smu = [np.linspace(1, 150, 151), np.linspace(-1, 1, 201)]
result_smu_avail = get_2PCF_from_LC(cutsky, cutsky_rd, edges=edges_smu, nthreads=256, mask=mask_tr & cutsky['AVAILABLE'], mask_rd=cutsky_rd['AVAILABLE'])

s_lc, (xi0_lc, xi2_lc) = result_smu_avail(return_sep=True, ells=[0,2])


result_smu_FA = get_2PCF_from_LC(cutsky, cutsky_rd, edges=edges_smu, nthreads=256, mask=mask_tr & (cutsky['NUMOBS']>0), mask_rd=cutsky_rd['AVAILABLE'], R1R2=result_smu_avail.R1R2)

s_fa, (xi0_fa, xi2_fa) = result_smu_FA(return_sep=True, ells=[0,2])

result_smu_FA_wcomp = get_2PCF_from_LC(cutsky, cutsky_rd, edges=edges_smu, nthreads=256, mask=mask_tr & (cutsky['NUMOBS']>0), mask_rd=cutsky_rd['AVAILABLE'], R1R2=result_smu_avail.R1R2, weights=cutsky['COMP_WEIGHT'])
s_fa_wcomp, (xi0_fa_wcomp, xi2_fa_wcomp) = result_smu_FA_wcomp(return_sep=True, ells=[0,2])


plt.plot(s_lc, xi0_lc*s_lc**2, label='Available for FA')
plt.plot(s_fa, xi0_fa*s_fa**2, label='Observed after FA')
plt.plot(s_fa_wcomp, xi0_fa_wcomp*s_fa_wcomp**2, label='Observed after FA w/ comp weight')

plt.legend()
plt.xlabel('s [Mpc/h]')
plt.ylabel('$s^2 \cdot \\xi_0$')


In [ ]:

edges_rppi = [np.geomspace(0.01, 100, 48), np.linspace(-40, 40, 81)]

result_rppi_avail = get_wp_from_LC(cutsky, cutsky_rd, edges=edges_rppi, nthreads=256, mask=mask_tr & cutsky['AVAILABLE'], mask_rd=cutsky_rd['AVAILABLE'])

rp_lc, wp_lc = result_rppi_avail(return_sep=True, pimax=40)


result_rppi_FA = get_wp_from_LC(cutsky, cutsky_rd, edges=edges_rppi, nthreads=256, mask=mask_tr & (cutsky['NUMOBS']>0), mask_rd=cutsky_rd['AVAILABLE'], R1R2=result_rppi_avail.R1R2)

rp_fa, wp_fa = result_rppi_FA(return_sep=True, pimax=40)

result_rppi_FA_wcomp = get_wp_from_LC(cutsky, cutsky_rd, edges=edges_rppi, nthreads=256, mask=mask_tr & (cutsky['NUMOBS']>0), mask_rd=cutsky_rd['AVAILABLE'], R1R2=result_rppi_avail.R1R2, weights=cutsky['COMP_WEIGHT'])
rp_fa_wcomp, wp_fa_wcomp = result_rppi_FA_wcomp(return_sep=True, pimax=40)


plt.semilogx(rp_lc, wp_lc*rp_lc, label='Available for FA')
plt.semilogx(rp_fa, wp_fa*rp_fa, label='Observed after FA')
plt.semilogx(rp_fa_wcomp, wp_fa_wcomp*rp_fa_wcomp, label='Observed after FA w/ comp weight')

plt.legend()
plt.xlabel('rp [Mpc/h]')
plt.ylabel('$r_p \cdot w_p$ [Mpc/$h$]')

